In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")

# Load and prepare the data
df = pd.read_csv("../data/diabetes.csv")

# Replace 0s in critical features with NaN
cols_to_replace = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_to_replace] = df[cols_to_replace].replace(0, np.NaN)
df.fillna(df.median(), inplace=True)

# Features and target
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Models
models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

# Store results
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1 Score": round(f1, 4)
    })

# Show results as a DataFrame
results_df = pd.DataFrame(results)
print("\n🔍 Model Comparison Table:\n")
print(results_df)



🔍 Model Comparison Table:

                 Model  Accuracy  Precision  Recall  F1 Score
0  Logistic Regression    0.7532     0.6667  0.6182    0.6415
1        Random Forest    0.7403     0.6316  0.6545    0.6429
2              XGBoost    0.7143     0.5873  0.6727    0.6271


In [2]:
# Save best model
import pickle
best_model = models["XGBoost"]
with open("../models/best_model.pkl", "wb") as f:
    pickle.dump(best_model, f)
